In [ ]:
!pip install -q langchain_community
!pip install -q unstructured

In [ ]:
%cd /content/drive/MyDrive/RAG
!pip install -q -r requirements.txt

/content/drive/MyDrive/RAG


In [ ]:
import json
import os
import subprocess
from langchain_community.document_loaders import UnstructuredHTMLLoader
from pathlib import Path
import base64
import http.client
from tqdm import tqdm
import requests

# 1. Raw Data → Connecting

- 데이터를 가져오는 단계
- txt 파일을 html로 로딩 및 원본 사이트 주소 매핑

LangChain을 활용해 로딩한 html은, 파일을 저장한 디렉토리의 주소를 metadata의 'source'로 가져오게 됩니다. 추후 답변 제공시, 디렉토리 주소가 아닌 실제 URL을 제공하기 위해, LangChain이 로딩한 데이터를 수정해야합니다.

이후, html 파일의 저장 경로가 담긴 'source'를, 이 json 파일과 연결해 실제 URL로 변환해줍니다.

**txt → html 변환 및 원본 사이트 주소 mapping**

In [ ]:
%cd /content/drive/MyDrive/RAG

url_to_filename_map = {}

# txt 파일에 정보를 가져오고 싶은 url 기입
with open("./my-urls.txt", "r") as file:
    urls = [url.strip() for url in file.readlines()]

folder_path = "rag_data"
if not os.path.exists(folder_path):
    os.makedirs(folder_path, exist_ok=True)

# html 파일 생성
for url in urls:
    filename = url.split("/")[-1] + ".html"
    file_path = os.path.join(folder_path, filename)
    subprocess.run(["wget", "-O", file_path, url], check=True)
    url_to_filename_map[url] = filename

# { <url> : <html 파일 저장경로> } 형태로 json 파일 생성
with open("url_to_filename_map.json", "w") as map_file:
    json.dump(url_to_filename_map, map_file)

print('url_to_filename_map : ', url_to_filename_map)

/content/drive/MyDrive/RAG
url_to_filename_map :  {'https://colab.google': 'colab.google.html', 'https://medium.com/google-colab': 'google-colab.html', 'https://colab.google/notebooks': 'notebooks.html'}


**LangChain 활용 HTML 로딩**

Langchain의 UnstructuredHTMLLoader를 사용하여 이후 문단 나누기와 임베딩을 위해 machine readable한 형태로 변환

- page_content: 모든 텍스트
- metadata: 메타데이터 (source 안에 저장 경로 기록됨)

In [ ]:
!pip install --upgrade nltk
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
# 폴더 이름에 맞게 수정
html_files_dir = Path('/content/drive/MyDrive/RAG/rag_data')

html_files = list(html_files_dir.glob("*.html"))

datas = []

for html_file in html_files:
    loader = UnstructuredHTMLLoader(str(html_file))
    document_data = loader.load()
    print(document_data)
    datas.append(document_data)
    print(f"Processed {html_file}")

[Document(page_content='Google Colaboratory\n\nColab is a hosted Jupyter Notebook service that requires no setup to use and provides free access to computing resources, including GPUs and TPUs. Colab is especially well suited to machine learning, data science, and education.\n\nOpen Colab New Notebook\n\nBlog\n\nNews and Guidance\n\nFeatures, updates, and best practices\n\nRead our blog\n\nExplore\n\nBrowse Notebooks\n\nCheck out our catalog of sample notebooks illustrating the power and flexiblity of Colab.\n\nBrowse notebooks\n\nTrack changes\n\nRead about product updates, feature additions, bug fixes and other release details.\n\nView release notes\n\nDive deeper\n\nCheck out these resources to learn more about Colab and its ever-expanding ecosystem.\n\nFind resources\n\nResponsibilities\n\nBuilding responsible AI for everyone\n\nWe’re working to develop artificial intelligence responsibly in order to benefit people and society.\n\nLearn more', metadata={'source': '/content/drive/My

**Mapping 정보를 활용해 'source'를 실제 URL로 대체**

metadata의 'source'에 실제 URL이 아닌 html 파일이 저장된 디렉토리 경로가 담긴 것을 확인할 수 있습니다.

Mapping을 통해 이 'source'를 실제 URL로 바꾸는 작업



In [ ]:
with open("url_to_filename_map.json", "r") as map_file:
    url_to_filename_map = json.load(map_file)

filename_to_url_map = {v: k for k, v in url_to_filename_map.items()}
print(filename_to_url_map)

# datas 리스트의 각 Document 객체의 'source' 수정
for doc_list in datas:
    for doc in doc_list:
        extracted_filename = doc.metadata["source"].split("/")[-1]
        if extracted_filename in filename_to_url_map:
            doc.metadata["source"] = filename_to_url_map[extracted_filename]
            print(doc)
        else:
            print(f"Warning: {extracted_filename}에 해당하는 URL을 찾을 수 없습니다.")

{'colab.google.html': 'https://colab.google', 'google-colab.html': 'https://medium.com/google-colab', 'notebooks.html': 'https://colab.google/notebooks'}
page_content='Google Colaboratory\n\nColab is a hosted Jupyter Notebook service that requires no setup to use and provides free access to computing resources, including GPUs and TPUs. Colab is especially well suited to machine learning, data science, and education.\n\nOpen Colab New Notebook\n\nBlog\n\nNews and Guidance\n\nFeatures, updates, and best practices\n\nRead our blog\n\nExplore\n\nBrowse Notebooks\n\nCheck out our catalog of sample notebooks illustrating the power and flexiblity of Colab.\n\nBrowse notebooks\n\nTrack changes\n\nRead about product updates, feature additions, bug fixes and other release details.\n\nView release notes\n\nDive deeper\n\nCheck out these resources to learn more about Colab and its ever-expanding ecosystem.\n\nFind resources\n\nResponsibilities\n\nBuilding responsible AI for everyone\n\nWe’re worki

In [ ]:
# 이중 리스트를 풀어서 하나의 리스트로 만드는 작업
datas_flattened = [item for sublist in datas for item in sublist]
datas_flattened

[Document(page_content='Google Colaboratory\n\nColab is a hosted Jupyter Notebook service that requires no setup to use and provides free access to computing resources, including GPUs and TPUs. Colab is especially well suited to machine learning, data science, and education.\n\nOpen Colab New Notebook\n\nBlog\n\nNews and Guidance\n\nFeatures, updates, and best practices\n\nRead our blog\n\nExplore\n\nBrowse Notebooks\n\nCheck out our catalog of sample notebooks illustrating the power and flexiblity of Colab.\n\nBrowse notebooks\n\nTrack changes\n\nRead about product updates, feature additions, bug fixes and other release details.\n\nView release notes\n\nDive deeper\n\nCheck out these resources to learn more about Colab and its ever-expanding ecosystem.\n\nFind resources\n\nResponsibilities\n\nBuilding responsible AI for everyone\n\nWe’re working to develop artificial intelligence responsibly in order to benefit people and society.\n\nLearn more', metadata={'source': 'https://colab.goo

In [ ]:
# print(datas_flattened[0].page_content)
print(datas_flattened[0])

page_content='Google Colaboratory\n\nColab is a hosted Jupyter Notebook service that requires no setup to use and provides free access to computing resources, including GPUs and TPUs. Colab is especially well suited to machine learning, data science, and education.\n\nOpen Colab New Notebook\n\nBlog\n\nNews and Guidance\n\nFeatures, updates, and best practices\n\nRead our blog\n\nExplore\n\nBrowse Notebooks\n\nCheck out our catalog of sample notebooks illustrating the power and flexiblity of Colab.\n\nBrowse notebooks\n\nTrack changes\n\nRead about product updates, feature additions, bug fixes and other release details.\n\nView release notes\n\nDive deeper\n\nCheck out these resources to learn more about Colab and its ever-expanding ecosystem.\n\nFind resources\n\nResponsibilities\n\nBuilding responsible AI for everyone\n\nWe’re working to develop artificial intelligence responsibly in order to benefit people and society.\n\nLearn more' metadata={'source': 'https://colab.google'}


# 2. Chunking

- 임베딩 모델이 처리할 수 있는 적당한 크기로 raw data를 나누기
- 임베딩 모델마다 한 번에 처리할 수 있는 토큰 수의 한계가 있음

clovastudio 문단 나누기 API
- 문장들간의 의미 유사도를 찾아 최적의 chunk 개수와 사용자가 원하는 1개 chunk의 크기(글자 수)를 직접 설정하여 문단을 나눌수 있음
- 후처리(postProcess = True)를 통해 chunk당 글자 수의 상한선과 하한선을 postProcessMaxSize와 postProcessMinSize로 조절

In [ ]:
# 각 문자를 구분하여 분할 (문장이 온전하게 유지되지 않음)
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator = '',         # ''를 기준으로 분할
    chunk_size = 500,       # 최대 500자의 텍스트가 하나의 청크에 포함
    chunk_overlap  = 100,   # 각 청크는 연결 부분에서 100자가 중복
    length_function = len,  # 문자열의 길이를 기반으로 청크의 길이 계산
)

for data in datas_flattened:
  texts = text_splitter.split_text(data.page_content)
  print(texts[0])

page_content='Google Colaboratory\n\nColab is a hosted Jupyter Notebook service that requires no setup to use and provides free access to computing resources, including GPUs and TPUs. Colab is especially well suited to machine learning, data science, and education.\n\nOpen Colab New Notebook\n\nBlog\n\nNews and Guidance\n\nFeatures, updates, and best practices\n\nRead our blog\n\nExplore\n\nBrowse Notebooks\n\nCheck out our catalog of sample notebooks illustrating the power and flexiblity of Colab.\n\nBrowse notebooks\n\nTrack changes\n\nRead about product updates, feature additions, bug fixes and other release details.\n\nView release notes\n\nDive deeper\n\nCheck out these resources to learn more about Colab and its ever-expanding ecosystem.\n\nFind resources\n\nResponsibilities\n\nBuilding responsible AI for everyone\n\nWe’re working to develop artificial intelligence responsibly in order to benefit people and society.\n\nLearn more' metadata={'source': 'https://colab.google'}
Googl

In [ ]:
'''
텍스트를 재귀적으로 분할하여 의미적으로 관련 있는 텍스트 조각들이 같이 있도록 하는 목적으로 설계
문자 리스트(['\n\n', '\n', ' ', ''])의 문자를 순서대로 사용하여 텍스트를 분할하며,
분할된 청크들이 설정된 chunk_size보다 작아질 때까지 이 과정을 반복
'''
# RecursiveCharacterTextSplitter: 문장이 온전하게 유지된 채로 나누기
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap  = 100,
    length_function = len,
)

texts_recursive = []
for data in datas_flattened:
  texts = text_splitter.split_text(data.page_content)
  texts_recursive.append(texts)
  print(texts[0])

Google Colaboratory

Colab is a hosted Jupyter Notebook service that requires no setup to use and provides free access to computing resources, including GPUs and TPUs. Colab is especially well suited to machine learning, data science, and education.

Open Colab New Notebook

Blog

News and Guidance

Features, updates, and best practices

Read our blog

Explore

Browse Notebooks

Check out our catalog of sample notebooks illustrating the power and flexiblity of Colab.

Browse notebooks
Homepage

Open in app

Sign inGet started

Experiment, Iterate, Collaborate

https://colab.research.google.com

Colab Tools for Google Sheets

Colab Tools for Google Sheets

Article describing two of our new tools for using Google Sheets with Colab: Sheets to Colab and Interactive Sheets

Rory Pilgrim

Aug 15

Colab Pro and Pro+ now available via Workspace!

Colab Pro and Pro+ now available via Workspace!
Learning Resources

Curated Notebooks

Here you'll find a series of instructive and educational noteb

In [ ]:
# 모델에서 사용하는 토크나이저를 기준으로 분할
# tiktoken 토크나이저를 기준으로 글자 수를 계산하여 분할
# !pip install -q tiktoken
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=600,
    chunk_overlap=200,
    encoding_name='cl100k_base'  # 텍스트를 토큰으로 변환하는 인코딩 방식
)

for data in datas_flattened:
  if isinstance(data, tuple):
    from langchain.schema import Document
    data = Document(page_content=data[0], metadata=data[1])
  docs = text_splitter.split_documents([data])
  print(docs[0].page_content)

Google Colaboratory

Colab is a hosted Jupyter Notebook service that requires no setup to use and provides free access to computing resources, including GPUs and TPUs. Colab is especially well suited to machine learning, data science, and education.

Open Colab New Notebook

Blog

News and Guidance

Features, updates, and best practices

Read our blog

Explore

Browse Notebooks

Check out our catalog of sample notebooks illustrating the power and flexiblity of Colab.

Browse notebooks

Track changes

Read about product updates, feature additions, bug fixes and other release details.

View release notes

Dive deeper

Check out these resources to learn more about Colab and its ever-expanding ecosystem.

Find resources

Responsibilities

Building responsible AI for everyone

We’re working to develop artificial intelligence responsibly in order to benefit people and society.

Learn more
Homepage

Open in app

Sign inGet started

Experiment, Iterate, Collaborate

https://colab.research.goog

# 3. Embedding

- 텍스트 데이터를 숫자로 이루어진 벡터로 변환
- 벡터 공간 내에서 텍스트 간의 유사성을 계산 하거나 텍스트 기반으로 자연어를 처리하기 위한 사전 작업
- 텍스트의 의미적인 정보를 보존하도록 설계되어 있어, 벡터 공간에서 가까이 위치한 텍스트들은 의미적으로도 유사한 것으로 간주됨


- 임베딩 모델
  - OpenAI: GPT와 같은 언어 모델을 통해 텍스트의 임베딩 벡터를 생성할 수 있는 API를 제공합니다.
  - Hugging Face: Transformers 라이브러리를 통해 다양한 오픈소스 임베딩 모델을 제공합니다.
  - Google: Gemini, Gemma 등 언어 모델에 적용되는 임베딩 모델을 제공합니다.


- 임베딩 메소드:
  - embed_documents: 이 메소드는 문서 객체의 집합을 입력으로 받아, 각 문서를 벡터 공간에 임베딩합니다. 주로 대량의 텍스트 데이터를 배치 단위로 처리할 때 사용됩니다.
  - embed_query: 이 메소드는 단일 텍스트 쿼리를 입력으로 받아, 쿼리를 벡터 공간에 임베딩합니다. 주로 사용자의 검색 쿼리를 임베딩하여, 문서 집합 내에서 해당 쿼리와 유사한 내용을 찾아내는 데 사용됩니다.

In [ ]:
!pip install -q sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.3/245.3 kB 6.2 MB/s eta 0:00:00


In [ ]:
# huggingface 모델에서 사전 훈련된 임베딩 모델

from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings_model = HuggingFaceEmbeddings(
    model_name='jhgan/ko-sroberta-nli',  # 한국어 자연어 추론 최적화 모델
    model_kwargs={'device':'cpu'},   # cuda
    encode_kwargs={'normalize_embeddings':True},  # 임베딩을 정규화하여 모든 벡터가 같은 범위의 값을 가지게 함
)

embeddings_model

# 임베딩 벡터는 768차원

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.49k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False}) with Transformer model: RobertaModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
), model_name='jhgan/ko-sroberta-nli', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': True}, multi_process=False, show_progress=False)

# 4. RAG - 백터 저장소

- 벡터 형태로 표현된 데이터(임베딩된 데이터)를 효율적으로 저장하고 검색할 수 있는 시스템
  - 대규모 벡터에서 빠른 속도로 가장 유사한 항목을 찾는것이 목표
  - 코사인 유사도, 유클리드 거리, 맨해튼 거리 등의 휴사도 측정 방법 사용

- 임베딩 벡터는 텍스트, 이미지, 소리 등 다양한 형태의 데이터를 벡터 공간에 매핑한 것으로, 데이터의 의미적, 시각적, 오디오적 특성을 수치적으로 표현

- 이러한 벡터를 효율적으로 저장하기 위해서는 고차원 벡터를 처리할 수 있도록 최적화된 데이터 저장 구조가 필요

- 벡터 저장소는 Faiss(Facebook AI Similarity Search), Chroma, Elasticsearch, Pinecone 등 다양한 오픈 소스 및 상용 솔루션이 있음

In [ ]:
# sentence-transformers: 임베딩 모델을 허깅페이스에서 다운로드 받기 위한 패키지
!pip install -q faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 42.3 MB/s eta 0:00:00


Faiss


- 벡터의 압축된 표현을 사용하여 메모리 사용량을 최소화하면서도 검색 속도를 극대화

- 유사도 기준
  - 'l2' (기본값): 유클리디안 거리를 기반. 두 벡터 간의 거리가 작을수록 더 유사하다고 평가
  - 'ip' (내적): 내적 기반 유사도 측정 방법. 두 벡터의 방향성이 얼마나 유사한지를 평가. 값이 클수록 더 유사하다고 판단
  - 'cosine': 코사인 유사도를 기반, 두 벡터의 각도가 작을수록 (즉, 방향이 더 유사할수록) 더 유사하다고 평가. 내적과 유사하지만, 벡터의 크기에 영향을 받지 않음


In [ ]:
# 벡터스토어 db 인스턴스를 생성
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_community.embeddings import HuggingFaceEmbeddings

# 문서 객체를 임베딩 벡터로 변환하여 벡터 저장소에 저장
vectorstore = FAISS.from_documents(datas_flattened,
                                   embedding = embeddings_model,
                                   distance_strategy = DistanceStrategy.COSINE
                                  )
vectorstore

In [ ]:
# 벡터 스토어 로컬 저장
vectorstore.save_local('./db/vectorstore')

In [ ]:
# 로컬 벡터 스토어 불러오기 (저장 시 사용된 임베딩 모델과 동인한 모델 사용해야 함)
mydb = FAISS.load_local('./db/vectorstore', embeddings_model, allow_dangerous_deserialization=True)
mydb

In [ ]:
datas_flattened

[Document(page_content='Google Colaboratory\n\nColab is a hosted Jupyter Notebook service that requires no setup to use and provides free access to computing resources, including GPUs and TPUs. Colab is especially well suited to machine learning, data science, and education.\n\nOpen Colab New Notebook\n\nBlog\n\nNews and Guidance\n\nFeatures, updates, and best practices\n\nRead our blog\n\nExplore\n\nBrowse Notebooks\n\nCheck out our catalog of sample notebooks illustrating the power and flexiblity of Colab.\n\nBrowse notebooks\n\nTrack changes\n\nRead about product updates, feature additions, bug fixes and other release details.\n\nView release notes\n\nDive deeper\n\nCheck out these resources to learn more about Colab and its ever-expanding ecosystem.\n\nFind resources\n\nResponsibilities\n\nBuilding responsible AI for everyone\n\nWe’re working to develop artificial intelligence responsibly in order to benefit people and society.\n\nLearn more', metadata={'source': 'https://colab.goo

In [ ]:
index = vectorstore.index
vector = index.reconstruct(0) # 첫번째 문서의 벡터 확인
print(vector)

[ 1.16752759e-02 -4.75443248e-03  3.25598085e-04  4.73441668e-02
 -7.51105845e-02 -2.70535257e-02 -2.04489641e-02 -3.76190408e-03
 -9.15825646e-03  1.52688939e-02  4.54313643e-02  2.36699209e-02
  4.55594435e-03  3.73931266e-02  2.82481331e-02  1.68343354e-02
  1.24867968e-02 -7.18616396e-02 -6.54376810e-03 -5.19109257e-02
  1.20540792e-02 -5.46796713e-03 -1.92015234e-03  8.28885473e-03
 -1.18723279e-03  8.03812779e-03  9.21870302e-03  3.25421952e-02
  5.98878413e-03  5.31751402e-02 -9.90301836e-03 -3.16669815e-03
 -2.74586771e-02  3.28996368e-02  4.68082465e-02 -3.90193774e-03
 -8.06775540e-02  4.81298976e-02 -3.48781496e-02 -2.08953079e-02
 -4.99950945e-02 -1.91320654e-03 -6.38581486e-03 -2.13441551e-02
 -4.30707149e-02 -7.18863541e-03 -7.89712276e-03 -8.61445162e-03
 -2.30783541e-02  1.68924090e-02  2.34455224e-02  2.40482492e-04
  1.81780588e-02  1.05495052e-02 -1.69635769e-02  3.21266279e-02
 -6.09201379e-02 -5.10859117e-02  2.99152210e-02 -4.03288491e-02
  4.35318500e-02  2.03943

문서 검색

주어진 쿼리 문자열에 대해 벡터 스토어 내의 문서들 중에서 가장 유사한 문서들을 찾아내기

- 쿼리 인코딩: 주어진 쿼리 문자열을 벡터로 변환합니다. 이 과정은 vectorstore를 생성할 때 사용된 임베딩 모델(embeddings_model)을 사용하여 수행됩니다.
- 유사도 검색: 변환된 쿼리 벡터와 벡터 스토어에 저장된 문서 벡터들 간의 유사도를 계산하여, 가장 유사한 문서들을 찾아냅니다. 이 때, 유사도 계산 방법은 vectorstore 생성 시 지정된 distance_strategy에 따라 결정됩니다.
- 결과 반환: 검색 결과로 얻어진 문서들을 유사도 순으로 정렬하여 반환합니다.

# 5. RAG - Retriever

- 벡터 저장소에서 문서를 검색하는 도구
- LangChain은 간단한 의미 검색도구부터 성능 향상을 위해 고려된 다양한 검색 알고리즘을 지원

In [ ]:
# 검색 쿼리
query = 'what is colab?'

# 가장 유사도가 높은 문장을 하나만 추출
retriever = vectorstore.as_retriever(search_kwargs={'k': 1})

docs = retriever.get_relevant_documents(query)
print(len(docs))
docs[0]
# docs = vectorstore.similarity_search(query)
# print(docs)
# print(docs[0].page_content)

1


Document(page_content='Homepage\n\nOpen in app\n\nSign inGet started\n\nExperiment, Iterate, Collaborate\n\nhttps://colab.research.google.com\n\nColab Tools for Google Sheets\n\nColab Tools for Google Sheets\n\nArticle describing two of our new tools for using Google Sheets with Colab: Sheets to Colab and Interactive Sheets\n\nRory Pilgrim\n\nAug 15\n\nColab Pro and Pro+ now available via Workspace!\n\nColab Pro and Pro+ now available via Workspace!\n\nWorkspace customers of all shapes and sizes now have the ability to buy Colab Pro and Pro+ licenses for users in their organization via the…\n\nJulianne DeMars-Smith\n\nJul 11\n\nColab Updated to Ubuntu 22.04 LTS\n\nColab Updated to Ubuntu 22.04 LTS\n\nWe’re happy to announce that Colab has upgraded its default runtime to Ubuntu 22.04 LTS. You can read more about the distribution changes…\n\nEric Johnson\n\nJul 19, 2023\n\nColab Data Visualizations Made Easy\n\nIf you’ve never found visualizing your data challenging, you can stop reading

In [ ]:
# MMR(Maximal Marginal Relevance) 검색
# MMR - 다양성 고려 (lambda_mult = 0.5)
retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 5, 'fetch_k': 50}  # 상위 5개 문서를 검색
)

docs = retriever.get_relevant_documents(query)
print(len(docs))
docs[1]


3


Document(page_content='Google Colaboratory\n\nColab is a hosted Jupyter Notebook service that requires no setup to use and provides free access to computing resources, including GPUs and TPUs. Colab is especially well suited to machine learning, data science, and education.\n\nOpen Colab New Notebook\n\nBlog\n\nNews and Guidance\n\nFeatures, updates, and best practices\n\nRead our blog\n\nExplore\n\nBrowse Notebooks\n\nCheck out our catalog of sample notebooks illustrating the power and flexiblity of Colab.\n\nBrowse notebooks\n\nTrack changes\n\nRead about product updates, feature additions, bug fixes and other release details.\n\nView release notes\n\nDive deeper\n\nCheck out these resources to learn more about Colab and its ever-expanding ecosystem.\n\nFind resources\n\nResponsibilities\n\nBuilding responsible AI for everyone\n\nWe’re working to develop artificial intelligence responsibly in order to benefit people and society.\n\nLearn more', metadata={'source': 'https://colab.goog

# 6. 답변 생성

- 벡터 저장소에서 문서를 검색한 다음, 이를 기반으로 ChatGPT 모델에 쿼리를 수행하는 end-to-end 프로세스를 구현

1. 검색 (Retrieval): vectorstore.as_retriever를 사용하여 MMR(Maximal Marginal Relevance) 검색 방식으로 문서를 검색합니다. search_kwargs에 k: 5와 lambda_mult: 0.15를 설정하여 상위 5개의 관련성이 높으면서도 다양한 문서를 선택합니다.

2. 프롬프트 생성 (Prompt): ChatPromptTemplate를 사용하여 쿼리에 대한 답변을 생성하기 위한 템플릿을 정의합니다. 여기서 {context}는 검색된 문서의 내용이고, {question}은 사용자의 쿼리입니다.

3. 모델 (Model): ChatOpenAI를 사용하여 OpenAI의 GPT 모델을 초기화합니다. 이 예에서는 'gpt-3.5-turbo-0125' 모델을 사용하며, temperature를 0으로 설정하여 결정론적인 응답을 생성하고, max_tokens를 500으로 설정하여 응답의 길이를 제한합니다.

4. 문서 포맷팅 (Formatting Docs): 검색된 문서(docs)를 포맷팅하는 format_docs 함수를 정의합니다. 이 함수는 각 문서의 page_content를 가져와 두 개의 문단 사이에 두 개의 줄바꿈을 삽입하여 문자열로 결합합니다.

5. 체인 실행 (Chain Execution): prompt | llm | StrOutputParser()를 사용하여 LLM 체인을 구성하고 실행합니다. 프롬프트를 통해 정의된 쿼리를 모델에 전달하고, 모델의 응답을 문자열로 파싱합니다.

6. 실행 (Run): chain.invoke 메서드를 사용하여 체인을 실행합니다. context로는 포맷팅된 문서 내용이고, question은 사용자의 쿼리입니다. 최종 응답은 response 변수에 저장됩니다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# Retrieval
retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 5, 'lambda_mult': 0.15}
)

docs = retriever.get_relevant_documents(query)

# Prompt
template = '''Answer the question based only on the following context:
{context}

Question: {question}
'''

prompt = ChatPromptTemplate.from_template(template)

# Model
llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0,
    max_tokens=500,
)


def format_docs(docs):
    return '\n\n'.join([d.page_content for d in docs])

# Chain
chain = prompt | llm | StrOutputParser()

# Run
response = chain.invoke({'context': (format_docs(docs)), 'question':query})
response
